In [3]:
import pandas as pd
import random

# =========================
# Parameters
# =========================
N = 10          # number of caissons
T = 30          # planning horizon
caissons = list(range(N))

ALPHA_MIN = 0.001
ALPHA_MAX = 0.05

INITIAL_P_MIN = 3.2
INITIAL_P_MAX = 3.5

# 每次 x 任务后，alpha 的增长百分比范围
GROWTH_MIN = 0.1   # 10%
GROWTH_MAX = 0.3   # 30%

SEED = 40


# =========================
# Generate base parameters
# =========================
random.seed(SEED)
alpha_values = {
    caisson: round(random.uniform(ALPHA_MIN, ALPHA_MAX), 6)
    for caisson in caissons
}

random.seed(SEED)
initial_p_values = {
    caisson: round(random.uniform(INITIAL_P_MIN, INITIAL_P_MAX), 2)
    for caisson in caissons
}


# =========================
# Task generation function
# =========================
def generate_tasks_variable_alpha(initial_p_values, T, alpha_values,
                                  growth_min=0.02, growth_max=0.08,
                                  seed=None):
    """
    Generate x tasks for each caisson.
    After each x task, alpha increases by a random percentage:
        alpha_next = alpha_now * (1 + growth)
    where growth ~ Uniform(growth_min, growth_max)
    """
    if seed is not None:
        random.seed(seed)

    columns = [
        'INDEX', 'CSEM', 'INITIALP', 'OPERATION', 'DATE',
        'ALPHA_BEFORE_X', 'GROWTH_RATE', 'ALPHA_AFTER_X'
    ]
    data = []

    index = 0

    for caisson, initial_p in initial_p_values.items():
        current_p = initial_p
        total_days = 0
        alpha = alpha_values[caisson]

        while total_days < T:
            # 避免异常情况
            if alpha <= 0:
                break

            # 从当前压力降到 3.2 所需时间
            time_to_3_2 = int((current_p - 3.2) / alpha)

            # 防止因为 very small gap 或浮点误差出现负数
            time_to_3_2 = max(0, time_to_3_2)

            task_day = total_days + time_to_3_2

            # 超出 horizon 就停止
            if task_day >= T:
                break

            alpha_before = alpha

            # 每次 x 后 alpha 随机按百分比增长
            growth_rate = random.uniform(growth_min, growth_max)
            alpha_after = alpha_before * (1 + growth_rate)

            # 记录本次 x 任务
            data.append([
                index,
                caisson,
                initial_p,
                'x',
                task_day,
                round(alpha_before, 6),
                round(growth_rate, 6),
                round(alpha_after, 6)
            ])

            index += 1

            # x task 占 1 天
            total_days = task_day + 1

            # 维修后压力恢复到 3.5
            current_p = 3.5

            # 更新 alpha，供下一轮使用
            alpha = alpha_after

    maintenance_plan = pd.DataFrame(data, columns=columns)
    return maintenance_plan


# =========================
# Run
# =========================
planning = generate_tasks_variable_alpha(
    initial_p_values=initial_p_values,
    T=T,
    alpha_values=alpha_values,
    growth_min=GROWTH_MIN,
    growth_max=GROWTH_MAX,
    seed=SEED
)

print("Initial alpha values:")
print(alpha_values)
print("\nInitial pressure values:")
print(initial_p_values)

print("\nGenerated maintenance plan:")
print(planning)

planning.to_csv("maintenance_plan_uncertainPercentage.csv", index=False)
print("\nCSV file has been saved successfully")

Initial alpha values:
{0: 0.023472, 1: 0.044016, 2: 0.00256, 3: 0.014839, 4: 0.048128, 5: 0.033552, 6: 0.00729, 7: 0.018071, 8: 0.044108, 9: 0.022597}

Initial pressure values:
{0: 3.34, 1: 3.46, 2: 3.21, 3: 3.28, 4: 3.49, 5: 3.4, 6: 3.24, 7: 3.3, 8: 3.46, 9: 3.33}

Generated maintenance plan:
    INDEX  CSEM  INITIALP OPERATION  DATE  ALPHA_BEFORE_X  GROWTH_RATE  \
0       0     0      3.34         x     5        0.023472     0.191721   
1       1     0      3.34         x    16        0.027972     0.275574   
2       2     0      3.34         x    25        0.035680     0.106369   
3       3     1      3.46         x     5        0.044016     0.156485   
4       4     1      3.46         x    11        0.050904     0.292358   
5       5     1      3.46         x    16        0.065786     0.232864   
6       6     1      3.46         x    20        0.081105     0.125674   
7       7     1      3.46         x    24        0.091298     0.169678   
8       8     1      3.46         x    